# Notebook 4: Inference Demo — Using Your Fine-Tuned Flan-T5

**What this notebook covers:**
- Load a model from Hugging Face Hub and run inference
- Compare baseline `google/flan-t5-base` vs your fine-tuned model side by side
- Experiment with `max_new_tokens` to control summary length
- Try the model on your own custom dialogues

**Connects to:** `inference/model.py` — the `predict()` function here is identical to production.

> **Note:** Set `FINETUNED_MODEL_ID` in Cell 3 to your HF Hub repo (from `training_demo.ipynb`).
> If you haven't fine-tuned yet, the notebook works fine with the baseline model only.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q "torch>=2.10.0" "transformers>=5.2.0" "huggingface_hub>=1.4.1" "datasets>=4.6.0"

In [ ]:
# Cell 2 — Configuration: set your model IDs here

# Replace with your fine-tuned model repo pushed in training_demo.ipynb
FINETUNED_MODEL_ID = "your-username/flan-t5-samsum-summarizer"

# The unmodified base model (always available, no login required)
BASELINE_MODEL_ID = "google/flan-t5-base"

## How inference works

In **transformers 5.x**, the old `pipeline('summarization')` was removed.
We use `AutoModelForSeq2SeqLM` and `model.generate()` directly — which is
exactly what `inference/model.py` does in production.

Three things to know:
1. **`"summarize: "` prefix** — Flan-T5 was instruction-tuned on many tasks, each with a text prefix. This prefix tells the model which task to perform.
2. **`max_new_tokens`** — sets an *upper bound* on the generated summary length (not a fixed length).
3. **`torch.no_grad()`** — disables gradient tracking at inference time to save memory and speed things up.

In [ ]:
# Cell 3 — Define predict() and load the baseline model
# This mirrors inference/model.py exactly
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def load_model_for_id(model_id):
    """Load tokenizer + model from HF Hub."""
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    model.eval()  # disable dropout, use deterministic inference
    return tokenizer, model

def predict(tokenizer, model, text, max_new_tokens=128):
    """Generate a summary for the given text."""
    # Add the task instruction prefix
    inputs = tokenizer(
        "summarize: " + text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Loading baseline model (google/flan-t5-base)...")
base_tok, base_model = load_model_for_id(BASELINE_MODEL_ID)
print("Baseline model ready.")

In [ ]:
# Cell 4 — Test baseline on 3 custom dialogues
custom_dialogues = [
    # Casual / social
    """Sam: Hey, are you free this weekend?
Alex: Saturday is packed but Sunday works.
Sam: Sunday afternoon? We could grab brunch.
Alex: Sounds perfect, let's say 11am at Maple Cafe.
Sam: Great, see you then!""",

    # Work / scheduling
    """Manager: The client call is moved to Thursday 3pm.
Dev: I have a conflict at 3, can we do 4pm instead?
Manager: Let me check... yes, 4pm works.
Dev: Perfect. Should I prepare a demo?
Manager: Yes, focus on the new dashboard features.
Dev: Got it, I'll have slides and a live demo ready.""",

    # Practical / logistics
    """Mia: Did the grocery delivery arrive?
Tom: Yes, but they substituted the almond milk with oat milk.
Mia: That's fine, I actually prefer oat milk.
Tom: Also they were out of sourdough, sent whole wheat instead.
Mia: Ugh, I really wanted sourdough for the weekend.
Tom: I can swing by the bakery on my way home.
Mia: That would be amazing, thanks!""",
]

print("=" * 65)
for i, dialogue in enumerate(custom_dialogues, 1):
    print(f"\nExample {i}:")
    print("INPUT:")
    print(dialogue)
    summary = predict(base_tok, base_model, dialogue)
    print(f"\nBASELINE SUMMARY:")
    print(summary)
    print("=" * 65)

In [ ]:
# Cell 5 — Load the fine-tuned model (skip gracefully if not set)
ft_tok, ft_model = None, None

if "your-username" in FINETUNED_MODEL_ID:
    print("\u26a0\ufe0f  Fine-tuned model not configured.")
    print("   Set FINETUNED_MODEL_ID in Cell 2 to enable the comparison.")
else:
    try:
        print(f"Loading fine-tuned model ({FINETUNED_MODEL_ID})...")
        ft_tok, ft_model = load_model_for_id(FINETUNED_MODEL_ID)
        print("Fine-tuned model ready.")
    except Exception as e:
        print(f"Could not load fine-tuned model: {e}")
        print("Continuing with baseline only.")

In [ ]:
# Cell 6 — Side-by-side comparison on 5 SAMSum test examples
import random
from datasets import load_dataset

random.seed(42)
test_ds = load_dataset("samsum", split="test")
samples = random.sample(list(test_ds), 5)

for i, ex in enumerate(samples, 1):
    print(f"\n{'='*65}")
    print(f"EXAMPLE {i}")
    print(f"{'='*65}")
    print("DIALOGUE:")
    print(ex["dialogue"])
    print("\nBASELINE SUMMARY:")
    print(predict(base_tok, base_model, ex["dialogue"]))
    if ft_model:
        print("\nFINE-TUNED SUMMARY:")
        print(predict(ft_tok, ft_model, ex["dialogue"]))
    else:
        print("\nFINE-TUNED SUMMARY: N/A (set FINETUNED_MODEL_ID in Cell 2)")
    print("\nREFERENCE SUMMARY:")
    print(ex["summary"])

print(f"\n{'='*65}")

In [ ]:
# Cell 7 — Effect of max_new_tokens on summary length
demo_dialogue = """Jordan: I just heard back from the university — I got in!
Casey: No way! That's incredible, congratulations!
Jordan: I can't believe it. The acceptance rate is like 8%.
Casey: You worked so hard for this. Which program?
Jordan: Computer Science, starting in September.
Casey: We need to celebrate. Dinner this weekend?
Jordan: Absolutely, you're buying! Kidding... maybe."""

print("Same dialogue, different max_new_tokens:\n")
print("DIALOGUE:")
print(demo_dialogue)
print()

for max_tokens in [32, 64, 128, 256]:
    result = predict(base_tok, base_model, demo_dialogue, max_new_tokens=max_tokens)
    word_count = len(result.split())
    print(f"max_new_tokens={max_tokens:>3} ({word_count:>2} words): {result}")

In [ ]:
# Cell 8 — ✏️ Interactive: change this dialogue and re-run!
my_dialogue = """
Alex: Did you finish the report?
Jordan: Almost, just need to add the charts.
Alex: We need it by 5pm today.
Jordan: I'll have it done by 3pm, no worries.
Alex: Perfect, thanks!
"""

print("Your dialogue:")
print(my_dialogue.strip())

print("\nBaseline summary:")
print(predict(base_tok, base_model, my_dialogue.strip()))

if ft_model:
    print("\nFine-tuned summary:")
    print(predict(ft_tok, ft_model, my_dialogue.strip()))

## Connection to the Production Codebase

The `predict()` function in this notebook is **identical** to `inference/model.py` — just
without the module-level singleton caching.

In production (`inference/model.py`):
- `_model` and `_tokenizer` are module-level globals
- `load_model()` is called once at startup (via the FastAPI `lifespan` context)
- Subsequent requests reuse the cached model — no repeated ~900MB downloads

```python
# inference/model.py — production version
_model = None
_tokenizer = None

def load_model():           # called once at API startup
    global _model, _tokenizer
    if _model is not None:
        return _tokenizer   # already loaded, return immediately
    ...
```

---

**Next:** In `05_api_walkthrough.ipynb` you will see how FastAPI wraps `predict()` into an HTTP
endpoint that can be called from anywhere — including the live App Runner deployment.